# 面试问题：怎样降低并评估幻觉，引用和拒答机制如何设计？

可直接复述的回答：先把回答拆成原子 claim，再逐条判断证据支持、矛盾或未知。引用必须指向真正支撑该 claim 的片段，不能只在段末放一个相关链接。检索分数不等于事实支持度，还要检查实体、数值、时间和否定词。证据不足时应拒答或请求澄清，并按风险设置不同阈值。评测至少包含 claim precision、citation correctness、coverage 和有用拒答率。线上要记录 claim-evidence 账本，便于定位是检索、生成还是门禁失败。

后续代码使用可读业务记录验证回答主线。实验结果均标注为教学实验，只用于解释机制，不代表真实线上收益。


## 1. 真实案例：智能手表支持文档与输入预览

五段脱敏产品文档保留版本、事实关键词和正文。案例覆盖防水、续航、充电、保修与停售型号；没有文档支持“太阳能充电”，用于拒答反例。


In [1]:
documents08 = [  # 构造带版本和事实关键词的支持文档。
    {"id": "manual-v3#water", "text": "Watch X 支持5ATM防水，但不建议潜水使用", "terms": {"watch x", "5atm", "防水", "不建议潜水"}},  # 防水规格证据。
    {"id": "manual-v3#battery", "text": "Watch X 标准模式续航最长36小时", "terms": {"watch x", "续航", "36小时"}},  # 续航规格证据。
    {"id": "manual-v3#charge", "text": "Watch X 仅支持磁吸充电底座", "terms": {"watch x", "磁吸充电", "仅支持"}},  # 充电方式证据。
    {"id": "warranty-2026", "text": "国行设备自激活日起提供一年有限保修", "terms": {"国行", "一年", "保修"}},  # 保修政策证据。
    {"id": "catalog-2026", "text": "Watch S 已停售，现有用户仍可获得安全更新", "terms": {"watch s", "停售", "安全更新"}},  # 老型号状态证据。
]  # 完成五段真实语义证据。
print("教学实验输入：产品支持证据")  # 标识输入预览。
for document08 in documents08:  # 逐条展示证据 ID 与正文。
    print(document08["id"], document08["text"])  # 输出可读文档片段。


教学实验输入：产品支持证据
manual-v3#water Watch X 支持5ATM防水，但不建议潜水使用
manual-v3#battery Watch X 标准模式续航最长36小时
manual-v3#charge Watch X 仅支持磁吸充电底座
warranty-2026 国行设备自激活日起提供一年有限保修
catalog-2026 Watch S 已停售，现有用户仍可获得安全更新


## 2. Baseline（基线）：总是引用最相关文档并回答

朴素方案按关键词重叠选一篇文档，即使重叠为零也声称“文档支持”。它把检索到相关产品和真正支持具体 claim 混为一谈。


In [2]:
claims08 = [  # 构造六条待验证原子 claim。
    {"id": "c1", "text": "Watch X 支持5ATM防水", "terms": {"watch x", "5atm", "防水"}, "gold": "supported"},  # 可被防水文档支持。
    {"id": "c2", "text": "Watch X 可以潜水", "terms": {"watch x", "潜水"}, "gold": "contradicted"},  # 与不建议潜水发生冲突。
    {"id": "c3", "text": "Watch X 续航36小时", "terms": {"watch x", "续航", "36小时"}, "gold": "supported"},  # 可被续航文档支持。
    {"id": "c4", "text": "Watch X 支持太阳能充电", "terms": {"watch x", "太阳能充电"}, "gold": "unknown"},  # 文档没有太阳能证据。
    {"id": "c5", "text": "国行设备保修一年", "terms": {"国行", "保修", "一年"}, "gold": "supported"},  # 可被保修政策支持。
    {"id": "c6", "text": "Watch S 已停售", "terms": {"watch s", "停售"}, "gold": "supported"},  # 可被目录支持。
]  # 完成可计算支持度的 claim 集。
baseline_rows08 = []  # 收集始终回答的基线结果。
for claim08 in claims08:  # 对每条 claim 选择重叠最多的文档。
    overlaps08 = [(len(claim08["terms"] & document08["terms"]), document08["id"]) for document08 in documents08]  # 计算关键词交集数量。
    best08 = max(overlaps08)  # 选择重叠最高的文档。
    baseline_rows08.append((claim08["id"], "supported", best08[1], best08[0], claim08["gold"]))  # 即使零证据也输出支持。
print("基线：claim | 预测 | 引用 | 重叠词数 | 真值")  # 输出始终回答方案表头。
for row08 in baseline_rows08:  # 逐条展示幻觉来源。
    print(row08)  # 输出基线 claim-evidence 对照。


基线：claim | 预测 | 引用 | 重叠词数 | 真值
('c1', 'supported', 'manual-v3#water', 3, 'supported')
('c2', 'supported', 'manual-v3#water', 1, 'contradicted')
('c3', 'supported', 'manual-v3#battery', 3, 'supported')
('c4', 'supported', 'manual-v3#water', 1, 'unknown')
('c5', 'supported', 'warranty-2026', 3, 'supported')
('c6', 'supported', 'catalog-2026', 2, 'supported')


## 3. 核心实现：原子 claim、支持度与矛盾规则

支持度使用 claim 关键词被同一文档覆盖的比例。涉及潜水时，证据中的“不建议潜水”触发矛盾；最高支持度低于 0.6 时拒答。真实系统应使用 NLI 或规则组合，这里保留可解释底层计算。


In [3]:
grounded_rows08 = []  # 收集逐 claim 的证据判定。
for claim08 in claims08:  # 逐条评估原子事实。
    scored08 = []  # 保存每篇文档对当前 claim 的覆盖度。
    for document08 in documents08:  # 遍历所有可见证据。
        score08 = len(claim08["terms"] & document08["terms"]) / len(claim08["terms"])  # 计算 claim 词项覆盖比例。
        scored08.append((score08, document08))  # 保存支持度与文档对象。
    best_score08, best_document08 = max(scored08, key=lambda item08: item08[0])  # 选择最强证据片段。
    contradiction08 = "潜水" in claim08["text"] and "不建议潜水" in best_document08["text"]  # 检测教学用否定冲突。
    if contradiction08:  # 优先处理明确矛盾。
        decision08 = "contradicted"  # 标记 claim 与证据冲突。
    elif best_score08 >= 0.60:  # 检查支持度是否达到回答阈值。
        decision08 = "supported"  # 接受 claim 并允许引用。
    else:  # 处理没有足够证据的情况。
        decision08 = "abstain"  # 拒绝把未知事实写成答案。
    citation08 = best_document08["id"] if decision08 != "abstain" else None  # 仅对支持或矛盾结论附证据。
    grounded_rows08.append((claim08["id"], round(best_score08, 2), decision08, citation08, claim08["gold"]))  # 保存完整 claim-evidence 账本。
print("核心过程：claim | 支持度 | 判定 | citation | 真值")  # 输出证据判定表头。
for row08 in grounded_rows08:  # 逐条展示支持度和引用。
    print(row08)  # 输出一个原子 claim 的证据账本。


核心过程：claim | 支持度 | 判定 | citation | 真值
('c1', 1.0, 'supported', 'manual-v3#water', 'supported')
('c2', 0.5, 'contradicted', 'manual-v3#water', 'contradicted')
('c3', 1.0, 'supported', 'manual-v3#battery', 'supported')
('c4', 0.5, 'abstain', None, 'unknown')
('c5', 1.0, 'supported', 'warranty-2026', 'supported')
('c6', 1.0, 'supported', 'catalog-2026', 'supported')


## 4. 结果表与结果解读

基线把所有 claim 都判为支持，因此在潜水和太阳能问题上产生幻觉。核心方案分别给出矛盾和拒答，并为可支持 claim 保留精确文档 ID。


In [4]:
baseline_correct08 = sum((row08[1] == row08[4]) or (row08[1] == "supported" and row08[4] == "supported") for row08 in baseline_rows08) / len(baseline_rows08)  # 计算始终回答的分类正确率。
normalized_grounded08 = [(row08[2] if row08[2] != "abstain" else "unknown") for row08 in grounded_rows08]  # 把拒答映射为未知真值标签。
grounded_correct08 = sum(prediction08 == claim08["gold"] for prediction08, claim08 in zip(normalized_grounded08, claims08)) / len(claims08)  # 计算门禁后的 claim 判定准确率。
citation_coverage08 = sum(row08[3] is not None for row08 in grounded_rows08) / len(grounded_rows08)  # 计算有证据结论的引用覆盖率。
print("方法 | claim准确率 | 引用覆盖率 | 未知处理")  # 输出方法对照表头。
print("always_answer", round(baseline_correct08, 3), 1.0, "错误声称支持")  # 展示基线幻觉问题。
print("ground_or_abstain", round(grounded_correct08, 3), round(citation_coverage08, 3), "拒答")  # 展示证据门禁结果。
print("结果解读：引用覆盖率下降不是退化，而是未知 claim 不伪造引用")  # 解释覆盖率和诚实拒答的关系。


方法 | claim准确率 | 引用覆盖率 | 未知处理
always_answer 0.667 1.0 错误声称支持
ground_or_abstain 1.0 0.833 拒答
结果解读：引用覆盖率下降不是退化，而是未知 claim 不伪造引用


## 5. 失败案例与修正：相关产品文档不等于支持太阳能

`c4` 与多篇 Watch X 文档共享产品名，基线会随便引用其中一篇。修正后要求具体事实词也被覆盖，最高支持度不足即拒答。


In [5]:
failure_baseline08 = next(row08 for row08 in baseline_rows08 if row08[0] == "c4")  # 读取太阳能 claim 的错误基线结果。
failure_fixed08 = next(row08 for row08 in grounded_rows08 if row08[0] == "c4")  # 读取证据门禁后的拒答结果。
print("失败行为", failure_baseline08)  # 展示无证据却附引用的回答。
print("修正行为", failure_fixed08)  # 展示低支持度触发拒答。
print("面向用户的修正文本：现有文档未说明太阳能充电，请确认型号或联系支持")  # 给出有用而非生硬的拒答。


失败行为 ('c4', 'supported', 'manual-v3#water', 1, 'unknown')
修正行为 ('c4', 0.5, 'abstain', None, 'unknown')
面向用户的修正文本：现有文档未说明太阳能充电，请确认型号或联系支持


## 6. 生产边界与 claim-evidence 账本

真实 claim 抽取、实体链接和矛盾识别远比词项覆盖复杂。还要执行 ACL、时间有效性、文档版本和引用片段定位，并人工检查高风险医疗、法律或金融结论。


In [6]:
grounding_contract08 = {"corpus": "watch-support-v3", "claim_extractor": "claim-v2", "support_threshold": 0.60, "citation_granularity": "section", "fallback": "clarify"}  # 定义可重放的证据门禁配置。
print("Grounding 发布合同", grounding_contract08)  # 展示文档版本与拒答阈值。
print("生产替换点：实体链接、NLI、ACL、有效时间、引用定位和高风险人工复核")  # 说明词项教学算法的局限。


Grounding 发布合同 {'corpus': 'watch-support-v3', 'claim_extractor': 'claim-v2', 'support_threshold': 0.6, 'citation_granularity': 'section', 'fallback': 'clarify'}
生产替换点：实体链接、NLI、ACL、有效时间、引用定位和高风险人工复核


## 7. 最小回归测试

断言保护真实案例规模、证据收益、矛盾和拒答反例。


In [7]:
assert len(documents08) >= 5 and len(claims08) >= 5  # 保证案例包含足够证据和 claim。
assert grounded_correct08 > baseline_correct08  # 保证证据门禁优于始终回答基线。
assert next(row08 for row08 in grounded_rows08 if row08[0] == "c2")[2] == "contradicted"  # 保证潜水冲突被识别。
assert failure_fixed08[2] == "abstain" and failure_fixed08[3] is None  # 保证太阳能未知事实不伪造引用。
print("最小回归测试通过：支持、矛盾、引用和拒答路径保持稳定")  # 显示 grounding 关键性质已验证。


最小回归测试通过：支持、矛盾、引用和拒答路径保持稳定
